In [1]:
from google.colab import files

# Завантажуємо processed_v2.csv
uploaded = files.upload()
file = list(uploaded.keys())[0]
print(f"Завантажено файл: {file}")
import pandas as pd

# Зчитуємо CSV
df = pd.read_csv(file)

# Перевіримо колонки
print(df.columns)
df.head(5)
print("Docs before filtering:", len(df))

texts = df["lemma_text"].astype(str)

Saving processed_v3_lemma.csv to processed_v3_lemma.csv
Завантажено файл: processed_v3_lemma.csv
Index(['sent_id', 'processed_text', 'lemma_text', 'upos_seq'], dtype='object')
Docs before filtering: 7142


In [20]:
# прибрати дуже короткі тексти
texts = texts[texts.str.split().str.len() > 5]

print("Docs after filtering:", len(texts))

Docs after filtering: 5338


In [21]:
custom_stopwords = [
    "це", "що", "як", "так", "не", "до", "на", "за",
    "ви", "ми", "вони", "цей", "той",
    "шановний", "колега", "депутат",
    "про", "питання", "увага", "будь", "ласка", "народний", "дякувати", "слово", "прошу",
    "фракція", "рада", "україна", "який", "бути"
]

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

TfidfVectorizer(
    min_df=5,
    max_df=0.9,
    stop_words=custom_stopwords
)

X_tfidf = tfidf.fit_transform(texts)

# k=5
lsa_5 = TruncatedSVD(n_components=5, random_state=42)
X_lsa_5 = lsa_5.fit_transform(X_tfidf)

# k=8
lsa_8 = TruncatedSVD(n_components=8, random_state=42)
X_lsa_8 = lsa_8.fit_transform(X_tfidf)

In [23]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

CountVectorizer(
    min_df=5,
    max_df=0.9,
    stop_words=custom_stopwords
)

X_count = count_vect.fit_transform(texts)

# k=5
lda_5 = LatentDirichletAllocation(n_components=5, random_state=42)
X_lda_5 = lda_5.fit_transform(X_count)

# k=8
lda_8 = LatentDirichletAllocation(n_components=8, random_state=42)
X_lda_8 = lda_8.fit_transform(X_count)

In [24]:
def print_topics(model, feature_names, n_top_words=10):
    topics = []
    for i, comp in enumerate(model.components_):
        words = [feature_names[j] for j in comp.argsort()[-n_top_words:][::-1]]
        topic_str = f"Topic {i}: {' '.join(words)}"
        print(topic_str)
        topics.append(words)
    return topics

In [25]:
feature_names_tfidf = tfidf.get_feature_names_out()
feature_names_count = count_vect.get_feature_names_out()

print("LSA k=5")
lsa5_topics = print_topics(lsa_5, feature_names_tfidf)

print("LDA k=5")
lda5_topics = print_topics(lda_5, feature_names_count)

LSA k=5
Topic 0: просити голосувати тому наш те закон сьогодні рішення верховний мати
Topic 1: просити голосувати передати підтримати зайняти голосування готовий заспокоїтися приготуватися місце
Topic 2: голосування закон проект ставити пропозиція верховний постанова та зміна внесення
Topic 3: володимир михайлович голосування ставити пропозиція михайло передати партія надаватися проект
Topic 4: голосувати хотіти голосування ще ставити сказати раз зараз готовий один
LDA k=5
Topic 0: те мати весь наш щоб російський сьогодні для влада мова
Topic 1: просити голосувати тому один рішення партія від володимир зараз перший
Topic 2: та рік для підготовка комунальний відсоток стан робота олімпійський проблема
Topic 3: закон верховний проект та щодо пропозиція постанова голосування президент зміна
Topic 4: хотіти ще сьогодні сказати по наш те такий якщо тому


In [26]:
import numpy as np

def get_top_docs(X_topics, texts, topic_idx, top_n=2):
    idx = np.argsort(X_topics[:, topic_idx])[::-1][:top_n]
    return texts.iloc[idx].values
print(get_top_docs(X_lda_5, texts, topic_idx=0))

['і , я думати , ми мусити і наш військовослужбовець , і їхній родина , і весь громадянин Україна , який російський , український , кримськотатарський мова говорити , мова незалежність Україна , мова територіальний цілісність і мова повага до наш український держава , подяка ви глибокий від наш фракція і від український парламент .'
 '2 жовтень в Москва відбутися консультація на рівень заступник міністр закордонний справа два країна , в хід який , однак , російський сторона , послатися на відсутність у вона інформація , не змогти надати відповідь на запитання стосовно ситуація з будівництво дамба і кінцевий мета будівництво .']
